# Test and fix a chat agent with simulated conversations

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/use-cases/end-to-end-agent-testing)

A Ridgeline Outfitters customer tells the support agent that a tent is torn, then asks what to do next. Does the agent remember the damage and send the customer to repairs, or give the returns address? You'll run the conversation, read the reply that failed, change one instruction, and check that ordinary returns still work.

## Set up the two processes

Use a Python 3.12 runtime with `venv`. The platform SDK exports policies; Agent Learning Kit simulates customers and checks the replies. They install overlapping `fi/` modules, so keep the two environments separate.

In [ ]:
%%bash -e
python3.12 -m venv .venv-platform
.venv-platform/bin/python -m pip install futureagi==0.8.0
python3.12 -m venv .venv-alk
.venv-alk/bin/python -m pip install agent-learning-kit==0.3.7 openai==2.54.0

In [ ]:
import os
from getpass import getpass

# Exported variables work in Jupyter; otherwise enter the keys at the hidden prompts.
for name in ['OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"Missing {name}")
        os.environ[name] = value
        del value

## 1. Snapshot the current instruction and the proposed fix

The baseline sends every issue to returns. The candidate sends damaged gear to repairs and preserves ordinary returns.

In [ ]:
%%writefile export_policies.py
import json
from pathlib import Path
from fi.prompt.types import PromptTemplate, SystemMessage

common = (
    "You are the support agent for Ridgeline Outfitters. "
    "Give one next step, including the contact email address. "
    "When the customer asks for the exact next step, reply with only the email address. "
)
policies = {
    "baseline": "For any gear issue, refer the customer to returns@ridgeline.example.",
    "candidate": (
        "For torn or damaged gear use repairs@ridgeline.example. "
        "For returns of unused gear use returns@ridgeline.example."
    ),
}
for name, instruction in policies.items():
    template = PromptTemplate(name=f"ridgeline-{name}",
                              messages=[SystemMessage(content=common + instruction)])
    message = template.messages[0].model_dump(include={"role", "content"})
    Path(f"{name}.json").write_text(json.dumps(message, indent=2))

In [ ]:
%%bash -e
.venv-platform/bin/python export_policies.py

### Optional: fetch committed versions

The next two cells use your workspace credentials to create a new practice prompt, commit both policies, and fetch them. Skip **both** cells to run the comparison with only local snapshots. No serving label is changed.

In [ ]:
import os
from getpass import getpass

# Exported variables work in Jupyter; otherwise enter the keys at the hidden prompts.
for name in ['FI_API_KEY', 'FI_SECRET_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"Missing {name}")
        os.environ[name] = value
        del value

In [ ]:
%%bash -e
export PROMPT_NAME="ridgeline-handoff-$(.venv-platform/bin/python -c 'import uuid; print(uuid.uuid4().hex)')"
export BASELINE_VERSION=v1 CANDIDATE_VERSION=v2
.venv-platform/bin/python - <<'PY'
import json
import os
from pathlib import Path
from fi.prompt import Prompt
from fi.prompt.types import ModelConfig, PromptTemplate, SystemMessage

def template(policy):
    snapshot = json.loads(Path(f"{policy}.json").read_text())
    return PromptTemplate(name=os.environ["PROMPT_NAME"],
                          messages=[SystemMessage(content=snapshot["content"])],
                          model_configuration=ModelConfig(model_name="gpt-4o-mini"))

prompt = Prompt(template=template("baseline"))
prompt.create()
prompt.commit_current_version(message="Baseline handoff policy")
prompt.create_new_version(template=template("candidate"))
prompt.commit_current_version(message="Route damaged gear to repairs")
PY

.venv-platform/bin/python - <<'PY'
import json
import os
from pathlib import Path
from fi.prompt import Prompt

for policy in ("baseline", "candidate"):
    prompt = Prompt.get_template_by_name(name=os.environ["PROMPT_NAME"],
                                         version=os.environ[f"{policy.upper()}_VERSION"])
    messages = prompt.template.model_dump()["messages"]
    if len(messages) != 1 or messages[0]["role"] != "system":
        raise ValueError("Expected one static system message")
    content = messages[0]["content"]
    if isinstance(content, list):
        if any(part.get("type") != "text" for part in content):
            raise ValueError("Expected text parts only")
        content = "".join(part["text"] for part in content)
    system = {"role": "system", "content": content}
    Path(f"{policy}.json").write_text(json.dumps(system, indent=2))
PY

## 2. Run two customers through the agent

Pass the full two-turn history to the model. The local check compares the entire final reply to the expected handoff address.

In [ ]:
%%writefile test_handoff.py
import asyncio
import json
import argparse
from pathlib import Path
from fi.evals import evaluate
from fi.simulate import AgentInput, Persona, Scenario, TestRunner
from openai import AsyncOpenAI

customers = [
    ("Ari", "The seam on my tent tore.", "repairs@ridgeline.example"),
    ("Jo", "I'd like to return an unused pack.", "returns@ridgeline.example"),
]
expected = {name: address for name, _, address in customers}
if len(expected) != len(customers):
    raise ValueError("Give each customer a unique case name")

scenario = Scenario(
    name="gear-handoff",
    dataset=[Persona(persona={"name": name}, situation=situation,
                     outcome="Find the right contact for my gear.")
             for name, situation, _ in customers],
)

async def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("policy", type=Path)
    parser.add_argument("--require-pass", action="store_true")
    args = parser.parse_args()
    policy = args.policy
    system = json.loads(policy.read_text())

    async with AsyncOpenAI() as client:
        async def reply(turn: AgentInput) -> str:
            response = await client.chat.completions.create(
                model="gpt-4o-mini", messages=[system, *turn.messages], temperature=0
            )
            content = response.choices[0].message.content
            if not content:
                raise RuntimeError("The model did not return a text reply")
            return content

        report = await TestRunner().run_test(scenario=scenario, agent_callback=reply,
                                             min_turns=2, max_turns=2)
    names = [case.persona.persona["name"] for case in report.results]
    if sorted(names) != sorted(expected):
        raise RuntimeError("Missing conversation; inspect the run before scoring")
    transcripts = []
    failures = 0
    for case in report.results:
        name = case.persona.persona["name"]
        replies = [m["content"] for m in case.messages if m["role"] == "assistant"]
        if len(replies) != 2:
            raise RuntimeError(f"Incomplete conversation for {name}; check the model call")
        result = evaluate("equals", engine="local", output=replies[-1],
                          expected_response=expected[name], config={"case_sensitive": True})
        if result.error or result.passed is None:
            raise RuntimeError(result.error or "Missing local eval verdict")
        failures += int(not result.passed)
        verdict = "PASS" if result.passed else "FAIL"
        print(f"{name}: {verdict} (expected {expected[name]})")
        transcripts.append(f"{name}: {verdict}\n{case.transcript}")
    Path(f"{policy.stem}-transcripts.txt").write_text("\n\n".join(transcripts))
    if args.require_pass and failures:
        raise SystemExit(f"{failures} handoff check(s) failed; inspect the transcript")

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
%%bash -e
.venv-alk/bin/python test_handoff.py baseline.json && cat baseline-transcripts.txt

## 3. Diagnose the failure and replay the same cases

Look for the address in Ari’s last reply. Run the candidate against both Ari and Jo; `--require-pass` fails the cell if either handoff is wrong. Each policy run makes four model calls.

In [ ]:
%%bash -e
.venv-alk/bin/python test_handoff.py candidate.json --require-pass && cat candidate-transcripts.txt

Read `baseline-transcripts.txt` and `candidate-transcripts.txt` in the notebook directory. A correct address checks the routing instruction, not whether a repair ticket was created.